# 02 — Ortholog Mapping

Map mouse gene symbols to human orthologs using the Ensembl REST API and a curated fallback table.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.config import *
from src.data_extraction import load_all_data
from src.ortholog_mapping import map_orthologs, mapping_summary, HARDCODED_ORTHOLOGS

In [2]:
data = load_all_data()
sig = data['sig_proteins']
mouse_genes = sig['mouse_gene'].tolist()
print(f'Genes to map: {len(mouse_genes)}')

[data_extraction] Loading real data from aba6334_data_file_s1.xlsx and aba6334_data_file_s3.xlsx


/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.replace({"undetected": np.nan, "NaN": np.nan, "nan": np.nan})
/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.replace({"undetected": np.nan, "NaN": np.nan, "nan": np.nan})
/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/d

  Proteins:      1505
  Significant:   71
  Mouse samples: 11
  Human subjects:58
  Data source:   supplement
Genes to map: 71


/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:371: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matrix[gene] = vals
/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:371: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matrix[gene] = vals
/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:371: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result o

In [4]:
# Map without API (fast, offline-safe)
cache_path = EXTERNAL_DIR / 'human_orthologs_cache.csv'
result_no_api = map_orthologs(mouse_genes, use_api=False, cache_path=cache_path)
print(f'Mapped (offline): {len(result_no_api)}')
result_no_api.head(10)

[ortholog_mapping] Saved mapping to /Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../data/external/human_orthologs_cache.csv
Mapped (offline): 71


,mouse_symbol,human_symbol,orthology_type,perc_id,confidence,source
0,Aplp1,APLP1,ortholog_one2one,100.0,1,hardcoded
1,Chl1,CHL1,ortholog_one2one,100.0,1,hardcoded
2,Mapt,MAPT,ortholog_one2one,100.0,1,hardcoded
3,App,APP,ortholog_one2one,100.0,1,hardcoded
4,Aplp2,APLP2,ortholog_one2one,100.0,1,hardcoded
5,Sort1,SORT1,ortholog_one2one,100.0,1,hardcoded
6,Ctsd,CTSD,ortholog_one2one,100.0,1,hardcoded
7,Sorl1,SORL1,ortholog_one2one,100.0,1,hardcoded
8,Ctsb,CTSB,ortholog_one2one,100.0,1,hardcoded
9,Cadm4,CADM4,ortholog_one2one,100.0,1,hardcoded


In [5]:
mapping_summary(result_no_api)

  Total:       71
  1:1 orthologs: 15 (21%)
  Hardcoded:   15
  Ensembl API: 0
  Fallback:    56


In [6]:
# Optional: run with Ensembl API (requires internet)
result_api = map_orthologs(mouse_genes, use_api=True, cache_path=cache_path)
mapping_summary(result_api)
print('(Ensembl API call commented out — set use_api=True to enable)')

[ortholog_mapping] Loaded 71 cached mappings; 0 new
[ortholog_mapping] Saved mapping to /Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../data/external/human_orthologs_cache.csv
  Total:       71
  1:1 orthologs: 15 (21%)
  Hardcoded:   15
  Ensembl API: 0
  Fallback:    56
(Ensembl API call commented out — set use_api=True to enable)


/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/ortholog_mapping.py:198: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat([cached, new_df], ignore_index=True)


In [7]:
# Save
result_no_api.to_csv(PROCESSED_DIR / 'human_orthologs.csv', index=False)
print('Saved to data/processed/human_orthologs.csv')
result_no_api.head()

Saved to data/processed/human_orthologs.csv


,mouse_symbol,human_symbol,orthology_type,perc_id,confidence,source
0,Aplp1,APLP1,ortholog_one2one,100.0,1,hardcoded
1,Chl1,CHL1,ortholog_one2one,100.0,1,hardcoded
2,Mapt,MAPT,ortholog_one2one,100.0,1,hardcoded
3,App,APP,ortholog_one2one,100.0,1,hardcoded
4,Aplp2,APLP2,ortholog_one2one,100.0,1,hardcoded


In [8]:
# Check key proteins
for gene in ['Aplp1', 'Chl1', 'Mapt', 'App', 'Sort1']:
    row = result_no_api[result_no_api['mouse_symbol'].str.lower() == gene.lower()]
    if len(row):
        print(f'  {gene} -> {row.iloc[0]["human_symbol"]}  ({row.iloc[0]["source"]})')
    else:
        print(f'  {gene} -> NOT MAPPED')

  Aplp1 -> APLP1  (hardcoded)
  Chl1 -> CHL1  (hardcoded)
  Mapt -> MAPT  (hardcoded)
  App -> APP  (hardcoded)
  Sort1 -> SORT1  (hardcoded)
